# 01 - What CIViC actually looks like

Every claim in `docs/spec-review.md` is reproduced here from the committed
snapshot. Run top to bottom; nothing here calls the network.

In [ ]:
import json, pathlib, sys, collections, statistics
sys.path.insert(0, '..')
SNAP = pathlib.Path('../data/snapshots/latest')
load = lambda n: [json.loads(l) for l in (SNAP / n).open()]
evidence   = load('evidence_items.jsonl')
profiles   = load('molecular_profiles.jsonl')
variants   = load('variants.jsonl')
assertions = load('assertions.jsonl')
print(f'{len(evidence):,} evidence  {len(profiles):,} profiles  '
      f'{len(variants):,} variants  {len(assertions)} assertions')

## The coordinate ceiling

The spec assumed a coordinate join. Only a minority of CIViC's variant
space carries coordinates at all, and the ones that do are GRCh37.

In [ ]:
coords = [v for v in variants if (v.get('coordinates') or {}).get('start')]
builds = collections.Counter((v['coordinates'].get('referenceBuild')) for v in coords)
print(f'variants with coordinates: {len(coords):,} / {len(variants):,} '
      f'({len(coords)/len(variants):.1%})')
print('reference build:', dict(builds))
print()
print('-> a GRCh38 callset joined without liftover matches ZERO of these,')
print('   and returns an empty set rather than an error.')

## What the variants without coordinates are

Not data gaps -- categories of variant that no coordinate can express.

In [ ]:
import re
nocoord = [v for v in variants if not (v.get('coordinates') or {}).get('start')]
kinds = collections.Counter()
for v in nocoord:
    name = (v.get('name') or '').upper()
    for kw in ('FUSION', '::', 'EXPRESSION', 'AMPLIFICATION', 'EXON',
               'MUTATION', 'DELETION', 'LOSS', 'METHYLATION'):
        if kw in name:
            kinds[kw] += 1
            break
    else:
        kinds['specific protein change'] += 1
for kind, count in kinds.most_common():
    print(f'{kind:26} {count:5,}')

## Evidence per profile, and why recall@10 is close to meaningless

In [ ]:
per_profile = collections.Counter(e['molecularProfile']['id'] for e in evidence)
sizes = sorted(per_profile.values())
print(f'profiles with >=1 evidence item: {len(per_profile):,} of {len(profiles):,}')
print(f'evidence per covered profile -- median {statistics.median(sizes):.0f}, '
      f'p90 {sizes[int(.9*len(sizes))]}, max {max(sizes)}')
print()
print('-> half of covered profiles have exactly one target. Recall@10 against')
print('   a single-item target measures prompt bloat, not retrieval quality.')

## Chunking is a non-issue

In [ ]:
lengths = sorted(len(e.get('description') or '') for e in evidence)
print(f'description length -- median {statistics.median(lengths):.0f} chars, '
      f'p90 {lengths[int(.9*len(lengths))]}, max {max(lengths)}')
print('-> one evidence item per chunk. Nothing to split, nothing to tune.')

## Why only 29 of 147 assertions are usable

In [ ]:
profile_by_id = {p['id']: p for p in profiles}
variant_by_id = {v['id']: v for v in variants}
reasons = collections.Counter()
for a in assertions:
    p = profile_by_id.get((a.get('molecularProfile') or {}).get('id', -1))
    if not p or len(p.get('variants') or []) != 1:
        reasons['multi-variant profile'] += 1
        continue
    v = variant_by_id.get(p['variants'][0]['id'], {})
    name = (v.get('name') or '').upper()
    if 'FUSION' in name or '::' in name or 'REARRANGE' in name:
        reasons['fusion / rearrangement'] += 1
    elif (v.get('coordinates') or {}).get('start'):
        reasons['usable'] += 1
    else:
        reasons['other non-coordinate'] += 1
for reason, count in reasons.most_common():
    print(f'{reason:26} {count:4}')
print()
print('-> the dominant blocker is fusions, which an SNV/indel VCF cannot express.')
print('   That is a ceiling on the task, not a bug in the matcher.')